# 3.2 Preprocessing

## 3.2.0 Train-test split
We split the training and testing set according to date, we split the data before 2026-05-25 as training set, the data after that as testing set.

In [7]:
import duckdb
from pathlib import Path

# import library and connect to db
con = duckdb.connect()

In [8]:
# load dataset
data_path = Path("../data/labeled.parquet")

df = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
""")

df.limit(5).df()

,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_240_normalized,smart_240_raw,smart_241_normalized,smart_241_raw,smart_242_normalized,smart_242_raw,days_to_failure,label_3d,label_7d,label_30d
0,2026-01-01,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
1,2026-01-02,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
2,2026-01-03,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
3,2026-01-04,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0
4,2026-01-05,1030A00UF97G,TOSHIBA MG07ACA14TA,14000519643136,0,sac2,000,1108,09,0013,...,100.0,0.0,NaN,NaN,NaN,NaN,<NA>,0,0,0


### Temporal Train-Test Split

In [9]:
cutoff = "2026-05-25"

train = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
    WHERE date < DATE '{cutoff}'
""")

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{data_path}')
    WHERE date >= DATE '{cutoff}'
""")

### verify split

In [10]:
print("Training Set:")
print(train.aggregate("COUNT(*) AS n, MIN(date) AS start, MAX(date) AS end").df())

print("\nTesting Set:")
print(test.aggregate("COUNT(*) AS n, MIN(date) AS start, MAX(date) AS end").df())

Training Set:
         n      start        end
0  1882420 2026-01-01 2026-05-24

Testing Set:
        n      start        end
0  479223 2026-05-25 2026-06-30


### Save training and testing set

In [11]:
con.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{data_path}')
        WHERE date < DATE '{cutoff}'
    )
    TO '../data/train.parquet' (FORMAT PARQUET)
""")

con.sql(f"""
    COPY (
        SELECT * FROM read_parquet('{data_path}')
        WHERE date >= DATE '{cutoff}'
    )
    TO '../data/test.parquet' (FORMAT PARQUET)
""")

CountBIGINT
479223
